In [ ]:
from dataclasses import asdict
import matplotlib.pyplot as plt
import pandas as pd

from utils.misc import resample_ohlcv
from strats.ict_oscillator_strategy import ICTOscillatorStrategy


In [ ]:

# Load pre-fetched 1m data from parquet
market_bars_df = pd.read_parquet("")
funding_df = pd.read_parquet("")

# Clean timestamp
funding_df["timestamp"] = funding_df["timestamp"].dt.floor("h")

symbol_data = {}
# market_bars_df['symbol'].unique()
for symbol in ['ETH']:
    df_1m_temp = market_bars_df[market_bars_df['symbol'] == symbol].reset_index(drop=True)
    df_funding_temp = funding_df[funding_df['symbol'] == symbol].reset_index(drop=True)
    
    df_1m_temp["timestamp"] = pd.to_datetime(df_1m_temp["timestamp"], utc=True)
    df_funding_temp["timestamp"] = pd.to_datetime(df_funding_temp["timestamp"], utc=True)
    
    
    symbol_data[symbol] = {
        "1m": df_1m_temp, 
        "15m": resample_ohlcv(df_1m_temp, "15min"), 
        "funding": df_funding_temp
    }

In [ ]:
strategy = ICTOscillatorStrategy()
start_time = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=2)
all_signals = []
for symbol, data_dict in symbol_data.items():
        df_1m = data_dict['1m']
        df_15m = data_dict['15m']
        df_funding = data_dict['funding'].astype({"timestamp": "datetime64[ns, UTC]", "funding_rate_relative": "float64"})
        df_15m_stochastic = strategy._build_15m_stochastic(df_15m)
        
        signals = strategy.generate_trade_signals(
            symbol=symbol, 
            df_1m=df_1m, 
            df_15m=df_15m_stochastic, 
            funding_df=df_funding,
            start_time=start_time
        )
        print(f"{symbol}: {len(signals)} signals")
        all_signals.extend(signals)  



In [ ]:
all_signals

In [ ]:
if all_signals:

    for sig in all_signals[:10]:
        df_1m = symbol_data[sig.symbol]['1m']
        df_15m = symbol_data[sig.symbol]['15m']
        df_funding = symbol_data[sig.symbol]['funding'].astype({"timestamp": "datetime64[ns, UTC]", "funding_rate_relative": "float64"})  # Ensure funding timestamps are timezone-aware
        df_15m_stochastic = strategy._build_15m_stochastic(df_15m)
        fig = strategy.plot_signal(
            sig, 
            df_1m=df_1m, 
            df_15m=df_15m_stochastic, 
            funding_df=df_funding, 
            return_fig=True, 
            lookahead_mins=60, 
            lookback_mins=60
        )
        plt.show(fig)
else:
    print("No signals found.")


In [ ]:
test = df_15m_stochastic[
    ["timestamp", "symbol", "K_line", "K_line_slow_stochastic", "D_line", "open"]
    ].set_index("timestamp").sort_index(ascending=True)

In [ ]:
df_merged = pd.merge_asof(
            df_1m,
            df_15m_stochastic[[
                "timestamp", "timestamp_15m",
                "K_line", "K_line_slow_stochastic", "D_line",
                "bullish_crossover", "bearish_crossover", "ema_close",
            ]],
            on="timestamp",
            direction="backward",
        ).set_index("timestamp").sort_index()

In [ ]:
df_merged.loc['2026-04-14 08:00': ]